# LeetCode 4073 — Count Good Strings (Hard)

Bài toán LeetCode:

```python
class Solution:
    def countGoodStrings(self, n: int) -> int:
        ...
```

Link tra cứu: https://leetcode.com/problemset/?search=4073 (Weekly Contest 522 — câu 4, rating khoảng 2125).

Một bài đếm tổ hợp mà `n` lên tới `10^15`! Hành trình giải bài này là một bài học mẫu mực về cách "giải mã" một định nghĩa đệ quy:

1. **Liệt kê bằng máy** các chuỗi tốt với `n` nhỏ, quan sát.
2. Phát hiện **đặc trưng đơn giản**: mọi "khối ký tự giống nhau liên tiếp" (run) đều có **độ dài lẻ**.
3. Đếm bằng **quy hoạch động** → nhận ra **dãy Fibonacci**.
4. Tính Fibonacci thứ `10^15` bằng **lũy thừa ma trận** hoặc **nhân đôi nhanh (fast doubling)** trong `O(log n)`.

---

## Sau notebook này, bạn sẽ nắm được

1. Đọc và cài đặt một định nghĩa đệ quy bằng memoization.
2. Phân tích chuỗi thành các **run** (`itertools.groupby`).
3. Chứng minh "chuỗi tốt ⇔ mọi run có độ dài lẻ".
4. Đếm số cách viết `n` thành tổng các số lẻ (có thứ tự) → Fibonacci.
5. Lũy thừa nhanh, lũy thừa ma trận `2×2`, fast doubling, phép lấy dư `10^9 + 7`.
6. Lỗi hay gặp, mẹo nhận diện, biến thể và bài tập.

# 1. Đề bài yêu cầu gì?

Xét các chuỗi chỉ gồm `'a'` và `'b'`. Một chuỗi được gọi là **tốt** nếu thỏa **một trong hai**:

1. Nó chỉ dùng **đúng một** loại ký tự và có **độ dài lẻ** (ví dụ `"a"`, `"bbb"`).
2. Nó tách được thành `s = s1 + s2`, với `s1`, `s2` đều là **chuỗi tốt khác rỗng**, và **ký tự cuối của `s1` khác ký tự đầu của `s2`**.

Cho `n`, đếm số chuỗi tốt độ dài `n`, lấy dư cho `10^9 + 7`.

---

## Ví dụ 1

```python
n = 4   ->   6
```

`"aaab"`, `"abbb"`, `"baaa"`, `"bbba"`, `"abab"`, `"baba"`.

- `"aaab" = "aaa" + "b"`: hai phần đều tốt (một loại ký tự, độ dài lẻ), ranh giới `a | b` khác nhau.
- `"ab" = "a" + "b"` là tốt, nên `"abab" = "ab" + "ab"` cũng tốt (ranh giới `b | a`).

## Ví dụ 2

```python
n = 3   ->   4
```

`"aaa"`, `"bbb"`, `"aba"`, `"bab"`.

## Ví dụ 3

```python
n = 2   ->   2
```

`"ab"`, `"ba"`.

---

## Ràng buộc

```text
1 <= n <= 10^15
```

# 2. Kiến thức nền tảng

## 2.1. Run — khối ký tự giống nhau liên tiếp

`"aaabba"` gồm các run `aaa`, `bb`, `a` với độ dài `3, 2, 1`. `itertools.groupby` tách run rất gọn.

In [1]:
from itertools import groupby, product


def run_lengths(s: str):
    return [len(list(g)) for _, g in groupby(s)]


print(run_lengths("aaabba"))   # [3, 2, 1]
print(run_lengths("abab"))     # [1, 1, 1, 1]

[3, 2, 1]
[1, 1, 1, 1]


## 2.2. Phép lấy dư `10^9 + 7`

Kết quả rất lớn nên đề yêu cầu lấy dư. Với phép cộng và nhân, ta được phép lấy dư **ở mọi bước trung gian**: `(a + b) % M = ((a % M) + (b % M)) % M`, tương tự với `*`.

## 2.3. Lũy thừa nhanh

`x^k` tính trong `O(log k)` phép nhân: nếu `k` chẵn thì `x^k = (x^(k/2))²`, nếu lẻ thì nhân thêm một `x`. Python có sẵn `pow(x, k, M)`.

In [2]:
MOD = 10**9 + 7


def fast_pow(x: int, k: int, m: int) -> int:
    result = 1
    x %= m
    while k:
        if k & 1:
            result = result * x % m
        x = x * x % m
        k >>= 1
    return result


print(fast_pow(3, 10**15, MOD), pow(3, 10**15, MOD))

468606845 468606845


# 3. Cách 1 — Brute Force: cài đặt đúng định nghĩa

Kiểm tra một chuỗi có tốt không bằng đệ quy + ghi nhớ (`lru_cache`): thử mọi điểm cắt. Rồi liệt kê mọi `2^n` chuỗi.

In [3]:
from functools import lru_cache


@lru_cache(maxsize=None)
def is_good(s: str) -> bool:
    if len(set(s)) == 1 and len(s) % 2 == 1:          # quy tắc 1
        return True
    for cut in range(1, len(s)):                       # quy tắc 2
        s1, s2 = s[:cut], s[cut:]
        if s1[-1] != s2[0] and is_good(s1) and is_good(s2):
            return True
    return False


def count_bruteforce(n: int) -> int:
    return sum(is_good("".join(p)) for p in product("ab", repeat=n))


for n in range(1, 11):
    print(n, count_bruteforce(n))

1 2
2 2
3 4
4 6
5 10
6 16
7 26
8 42
9 68
10 110


Dãy `2, 2, 4, 6, 10, 16, 26, 42, 68, 110` — mỗi số bằng tổng hai số trước! Đó là `2 ×` dãy Fibonacci (`1, 1, 2, 3, 5, 8, ...`). Nhưng trước khi tin, ta cần **hiểu vì sao**.

---

# 4. Bước đột phá 1: chuỗi tốt ⇔ mọi run có độ dài lẻ

Liệt kê các chuỗi tốt với `n = 5` cùng độ dài các run:

In [4]:
for p in product("ab", repeat=5):
    s = "".join(p)
    if is_good(s):
        print(s, run_lengths(s))

aaaaa [5]
aaaba [3, 1, 1]
abaaa [1, 1, 3]
ababa [1, 1, 1, 1, 1]
abbba [1, 3, 1]
baaab [1, 3, 1]
babab [1, 1, 1, 1, 1]
babbb [1, 1, 3]
bbbab [3, 1, 1]
bbbbb [5]


Mọi run đều lẻ! Chứng minh hai chiều:

**(⇐) Mọi run lẻ thì chuỗi tốt.** Quy nạp theo số run. Một run lẻ → tốt theo quy tắc 1. Nhiều run: tách run cuối ra, `s = s1 + (run cuối)`. `s1` có ít run hơn, mọi run lẻ → tốt; run cuối tốt; ranh giới giữa hai run khác ký tự → `s` tốt theo quy tắc 2.

**(⇒) Chuỗi tốt thì mọi run lẻ.** Quy nạp theo độ dài. Quy tắc 1 cho đúng một run lẻ. Quy tắc 2: `s = s1 + s2` với ký tự hai bên ranh giới **khác nhau** → ranh giới đó cũng là ranh giới giữa hai run; run của `s` chính là run của `s1` cộng run của `s2`, đều lẻ (giả thiết quy nạp).

Kiểm chứng bằng máy:

In [5]:
for n in range(1, 13):
    for p in product("ab", repeat=n):
        s = "".join(p)
        assert is_good(s) == all(r % 2 == 1 for r in run_lengths(s))
print("is_good(s) == (mọi run lẻ) với mọi chuỗi độ dài <= 12")

is_good(s) == (mọi run lẻ) với mọi chuỗi độ dài <= 12


# 5. Đếm: chia `n` thành các phần lẻ

Một chuỗi có mọi run lẻ được xác định bởi:

- **ký tự đầu tiên** (`a` hoặc `b`) — 2 cách; các run sau tự động xen kẽ ký tự;
- **dãy độ dài các run** — một cách viết `n = r1 + r2 + ... + rk` với mọi `ri` lẻ, **có thứ tự** (gọi là một **phân hoạch có thứ tự thành số lẻ** — composition).

Gọi `C(n)` = số cách viết `n` thành tổng có thứ tự các số lẻ. Đáp án = `2 · C(n)`.

## Cách 2 — DP `O(n²)`

Run đầu tiên có độ dài lẻ `r`, phần còn lại là một cách viết `n - r`:

```text
C(0) = 1,   C(n) = Σ_{r lẻ, r <= n} C(n - r)
```

In [6]:
def count_dp_quadratic(n: int) -> int:
    C = [0] * (n + 1)
    C[0] = 1
    for m in range(1, n + 1):
        C[m] = sum(C[m - r] for r in range(1, m + 1, 2)) % MOD
    return 2 * C[n] % MOD


print([count_dp_quadratic(n) for n in range(1, 11)])

[2, 2, 4, 6, 10, 16, 26, 42, 68, 110]


## Cách 3 — Rút gọn thành Fibonacci: `O(n)`

So sánh `C(n)` và `C(n - 2)`:

```text
C(n)     = C(n-1) + C(n-3) + C(n-5) + ...
C(n - 2) =          C(n-3) + C(n-5) + ...
```

Trừ hai dòng: `C(n) = C(n - 1) + C(n - 2)` — **công thức Fibonacci**! Với `C(1) = 1`, `C(2) = 1` (chỉ `1 + 1`), ta có `C(n) = F(n)` (Fibonacci với `F(1) = F(2) = 1`).

Cách nhìn trực tiếp: xét run cuối cùng. Nếu nó dài đúng 1 → bỏ nó đi, còn một cách viết của `n - 1`. Nếu nó dài `>= 3` → giảm nó đi 2, vẫn lẻ, được một cách viết của `n - 2`. Hai trường hợp không giao nhau và phủ hết.

```text
answer(n) = 2 · F(n) mod (10^9 + 7)
```

In [7]:
def count_linear(n: int) -> int:
    a, b = 1, 1                    # F(1), F(2)
    for _ in range(n - 1):
        a, b = b, (a + b) % MOD
    return 2 * a % MOD


print([count_linear(n) for n in range(1, 11)])

[2, 2, 4, 6, 10, 16, 26, 42, 68, 110]


`O(n)` với `n = 10^15` vẫn là quá nhiều vòng lặp. Cần `O(log n)`.

---

# 6. Cách 4 — Lũy thừa ma trận: O(log n)

Phép chuyển `(F(k+1), F(k)) → (F(k+2), F(k+1))` là một phép nhân ma trận:

```text
| F(k+2) |   | 1 1 |   | F(k+1) |
| F(k+1) | = | 1 0 | · | F(k)   |
```

Do đó:

```text
| 1 1 |^n   | F(n+1)  F(n)   |
| 1 0 |   = | F(n)    F(n-1) |
```

Lũy thừa ma trận `2×2` bằng **bình phương liên tiếp** (giống lũy thừa nhanh) → `O(log n)` phép nhân ma trận.

In [8]:
def mat_mult(A, B):
    return [[(A[0][0] * B[0][0] + A[0][1] * B[1][0]) % MOD, (A[0][0] * B[0][1] + A[0][1] * B[1][1]) % MOD],
            [(A[1][0] * B[0][0] + A[1][1] * B[1][0]) % MOD, (A[1][0] * B[0][1] + A[1][1] * B[1][1]) % MOD]]


def mat_pow(M, k):
    result = [[1, 0], [0, 1]]                  # ma trận đơn vị
    while k:
        if k & 1:
            result = mat_mult(result, M)
        M = mat_mult(M, M)
        k >>= 1
    return result


def count_matrix(n: int) -> int:
    F_n = mat_pow([[1, 1], [1, 0]], n)[0][1]
    return 2 * F_n % MOD


print([count_matrix(n) for n in range(1, 11)])
print("n = 10^15 ->", count_matrix(10**15))

[2, 2, 4, 6, 10, 16, 26, 42, 68, 110]
n = 10^15 -> 296650267


## Mô phỏng bình phương liên tiếp với `n = 10 = 1010₂`

| bit (từ phải) | M (lũy thừa) | bit = 1? | result |
|---|---|:---:|---|
| 0 | `T^1` | | `I` |
| 1 | `T^2` | ✔ | `T^2` |
| 2 | `T^4` | | `T^2` |
| 3 | `T^8` | ✔ | `T^10` |

`T^10 = [[F(11), F(10)], [F(10), F(9)]] = [[89, 55], [55, 34]]` → đáp án `2 · 55 = 110` ✔ (khớp brute force).

In [9]:
def trace_mat_pow(n: int):
    M, result, bit = [[1, 1], [1, 0]], [[1, 0], [0, 1]], 0
    k = n
    while k:
        if k & 1:
            result = mat_mult(result, M)
        print(f"bit {bit}: {'1' if k & 1 else '0'} -> result = {result}")
        M = mat_mult(M, M)
        k >>= 1
        bit += 1
    print(f"F({n}) = {result[0][1]}, đáp án = {2 * result[0][1] % MOD}")


trace_mat_pow(10)

bit 0: 0 -> result = [[1, 0], [0, 1]]
bit 1: 1 -> result = [[2, 1], [1, 1]]
bit 2: 0 -> result = [[2, 1], [1, 1]]
bit 3: 1 -> result = [[89, 55], [55, 34]]
F(10) = 55, đáp án = 110


# 7. Cách 5 — Fast doubling: O(log n), ít phép nhân hơn

Hai đẳng thức của Fibonacci (suy ra từ lũy thừa ma trận):

```text
F(2k)   = F(k) · (2·F(k+1) - F(k))
F(2k+1) = F(k)² + F(k+1)²
```

Từ cặp `(F(k), F(k+1))` tính được cặp `(F(2k), F(2k+1))`, rồi dịch thêm một bước nếu cần. Đi theo các bit của `n` từ cao xuống thấp.

In [10]:
def fib_pair(n: int):
    # trả về (F(n), F(n+1)) mod MOD, F(0) = 0
    if n == 0:
        return 0, 1
    a, b = fib_pair(n >> 1)                     # (F(k), F(k+1)), k = n // 2
    c = a * ((2 * b - a) % MOD) % MOD           # F(2k)
    d = (a * a + b * b) % MOD                   # F(2k+1)
    return (d, (c + d) % MOD) if n & 1 else (c, d)


def count_doubling(n: int) -> int:
    return 2 * fib_pair(n)[0] % MOD


print([count_doubling(n) for n in range(1, 11)])
print("n = 10^15 ->", count_doubling(10**15))

[2, 2, 4, 6, 10, 16, 26, 42, 68, 110]
n = 10^15 -> 296650267


Độ sâu đệ quy chỉ khoảng `log₂(10^15) ≈ 50` — an toàn.

# 8. Phiên bản đúng format LeetCode

In [11]:
class Solution:
    def countGoodStrings(self, n: int) -> int:
        MOD = 10**9 + 7

        def fib_pair(k):
            if k == 0:
                return 0, 1
            a, b = fib_pair(k >> 1)
            c = a * ((2 * b - a) % MOD) % MOD
            d = (a * a + b * b) % MOD
            return (d, (c + d) % MOD) if k & 1 else (c, d)

        return 2 * fib_pair(n)[0] % MOD

In [12]:
sol = Solution()
tests = [(4, 6, "ví dụ 1"), (3, 4, "ví dụ 2"), (2, 2, "ví dụ 3"), (1, 2, "n = 1: 'a', 'b'"), (10, 110, "n = 10")]
for n, expected, name in tests:
    result = sol.countGoodStrings(n)
    assert result == expected, (name, result, expected)
    print(f"PASS | {name:18} -> {result}")
print("n = 10^15 ->", sol.countGoodStrings(10**15))

PASS | ví dụ 1            -> 6
PASS | ví dụ 2            -> 4
PASS | ví dụ 3            -> 2
PASS | n = 1: 'a', 'b'    -> 2
PASS | n = 10             -> 110
n = 10^15 -> 296650267


# 9. Vì sao đúng?

1. **Đặc trưng** (mục 4): chuỗi tốt ⇔ mọi run có độ dài lẻ (chứng minh quy nạp hai chiều, kiểm chứng bằng máy tới `n = 12`).
2. **Song ánh**: chuỗi mọi run lẻ ↔ (ký tự đầu, dãy độ dài run lẻ có tổng `n`) → `2 · C(n)`.
3. **Fibonacci**: `C(n) = C(n-1) + C(n-2)`, `C(1) = C(2) = 1` → `C(n) = F(n)`.
4. **Tính nhanh**: lũy thừa ma trận / fast doubling đúng theo các đẳng thức Fibonacci; lấy dư ở mọi bước không đổi kết quả cuối mod `10^9 + 7`.

# 10. Độ phức tạp

| Cách | Ý tưởng | Thời gian | Bộ nhớ |
|---|---|---:|---:|
| 1. Brute Force | liệt kê `2^n` chuỗi, kiểm tra đệ quy | `O(2^n · n³)` | lớn |
| 2. DP tổng lẻ | `C(n) = Σ C(n - r)` | `O(n²)` | `O(n)` |
| 3. Fibonacci tuyến tính | `C(n) = C(n-1) + C(n-2)` | `O(n)` | `O(1)` |
| 4. Lũy thừa ma trận | `T^n` | `O(log n)` | `O(1)` |
| 5. Fast doubling | `F(2k)`, `F(2k+1)` | `O(log n)` | `O(log n)` (đệ quy) |

# 11. Đối chiếu năm cách

In [13]:
import random

for n in range(1, 13):
    expected = count_bruteforce(n)
    assert count_dp_quadratic(n) == expected
    assert count_linear(n) == expected
    assert count_matrix(n) == expected
    assert count_doubling(n) == expected
    assert sol.countGoodStrings(n) == expected
random.seed(4073)
for _ in range(200):
    n = random.randint(1, 3000)
    assert count_linear(n) == count_matrix(n) == count_doubling(n) == sol.countGoodStrings(n)
for _ in range(200):
    n = random.randint(1, 10**15)
    assert count_matrix(n) == count_doubling(n)
print("Brute force khớp tới n = 12; các cách nhanh khớp nhau trên 400 giá trị n ngẫu nhiên")

Brute force khớp tới n = 12; các cách nhanh khớp nhau trên 400 giá trị n ngẫu nhiên


# 12. Những lỗi dễ mắc

## Lỗi 1 — Quên nhân 2 (chỉ đếm chuỗi bắt đầu bằng `a`)

In [14]:
print("Sai :", fib_pair(4)[0], "| Đúng:", sol.countGoodStrings(4))

Sai : 3 | Đúng: 6


## Lỗi 2 — Lệch chỉ số Fibonacci

Trả về `F(n + 1)` hoặc `F(n - 1)` thay vì `F(n)`. Luôn kiểm tra với các giá trị nhỏ của đề: `n = 2 → 2`, `n = 3 → 4`, `n = 4 → 6`.

In [15]:
print("Sai (F(n+1)):", [2 * fib_pair(n + 1)[0] % MOD for n in (2, 3, 4)], "| Đúng:", [sol.countGoodStrings(n) for n in (2, 3, 4)])

Sai (F(n+1)): [4, 6, 10] | Đúng: [2, 4, 6]


## Lỗi 3 — Hiểu nhầm là "độ dài chuỗi lẻ" hoặc "số run lẻ"

Chuỗi `"abab"` có độ dài chẵn, 4 run — vẫn tốt. Điều kiện đúng: **mỗi** run có độ dài lẻ.

## Lỗi 4 — Vòng lặp `O(n)` với `n = 10^15`

Không bao giờ xong. Thấy `n` tới `10^15`/`10^18` trong bài đếm → nghĩ ngay tới **công thức truy hồi tuyến tính + lũy thừa ma trận**.

## Lỗi 5 — Số âm khi lấy dư trong fast doubling

`2·F(k+1) - F(k)` có thể **âm** sau khi đã lấy dư từng số. Python tự xử lý `%` cho số âm đúng, nhưng trong C++/Java phải cộng thêm `MOD` trước khi lấy dư.

# 13. Mẹo nhận diện

1. Định nghĩa đệ quy lạ → **liệt kê bằng máy** với `n` nhỏ, tìm **đặc trưng** đơn giản, rồi chứng minh.
2. Chuỗi xây từ các **khối** với ràng buộc độ dài → đếm **composition** (phân hoạch có thứ tự).
3. Dãy số thỏa `a(n) = a(n-1) + a(n-2)` (hoặc tổ hợp tuyến tính cố định) → **lũy thừa ma trận** cho `n` khổng lồ.
4. Tra dãy số đầu (2, 2, 4, 6, 10, 16, ...) — nhận ra Fibonacci nhân 2.

Các bài liên quan: *Climbing Stairs* (LeetCode 70), *Fibonacci Number* (509), *Count Vowels Permutation* (1220), *Knight Dialer* (935 — truy hồi tuyến tính, lũy thừa ma trận).

# 14. Biến thể

## Biến thể 1 — Mọi run có độ dài **chẵn**

`D(n)` = số cách viết `n` thành tổng các số chẵn = số cách viết `n/2` thành tổng số nguyên dương = `2^(n/2 - 1)` khi `n` chẵn, `0` khi lẻ. Đáp án `2 · D(n)`.

In [16]:
def count_all_even_runs(n: int) -> int:
    if n % 2:
        return 0
    return 2 * pow(2, n // 2 - 1, MOD) % MOD


def brute_even_runs(n):
    return sum(all(r % 2 == 0 for r in run_lengths("".join(p))) for p in product("ab", repeat=n))


for n in range(1, 11):
    assert count_all_even_runs(n) == brute_even_runs(n)
print([count_all_even_runs(n) for n in range(1, 11)])

[0, 2, 0, 4, 0, 8, 0, 16, 0, 32]


## Biến thể 2 — Bảng chữ cái `k` ký tự

Ký tự đầu có `k` cách; mỗi run sau phải khác ký tự của run ngay trước nên có `k - 1` cách. Khi đó số chuỗi không còn chỉ phụ thuộc dãy độ dài run mà cả số run, nhưng vẫn có một truy hồi bậc 2 gọn gàng — xem bài tập 6.

# 15. Template tư duy

```python
# 1. liệt kê nhỏ -> đoán đặc trưng -> chứng minh
# 2. đếm cấu trúc -> công thức truy hồi tuyến tính a(n) = c1·a(n-1) + c2·a(n-2) + ...
# 3. n khổng lồ -> lũy thừa ma trận (hoặc fast doubling cho Fibonacci), lấy dư mỗi bước
```

# 16. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Liệt kê mọi cách viết `6` thành tổng có thứ tự các số lẻ. Có bao nhiêu? Đáp án của đề với `n = 6` là bao nhiêu? Kiểm tra bằng cell dưới.

In [17]:
# Kiểm tra Bài 1
def compositions_odd(n):
    if n == 0:
        return [[]]
    return [[r] + rest for r in range(1, n + 1, 2) for rest in compositions_odd(n - r)]


print(compositions_odd(6), len(compositions_odd(6)), "| đáp án:", sol.countGoodStrings(6))

[[1, 1, 1, 1, 1, 1], [1, 1, 1, 3], [1, 1, 3, 1], [1, 3, 1, 1], [1, 5], [3, 1, 1, 1], [3, 3], [5, 1]] 8 | đáp án: 16


## Bài 2 — Dự đoán output

```python
print(run_lengths("abba"), is_good("abba"))
print(mat_pow([[1, 1], [1, 0]], 5))
print(fib_pair(10))
```

In [18]:
print(run_lengths("abba"), is_good("abba"))
print(mat_pow([[1, 1], [1, 0]], 5))
print(fib_pair(10))

[1, 2, 1] False
[[8, 5], [5, 3]]
(55, 89)


## Bài 3 — Tự viết lũy thừa ma trận cho Fibonacci

In [19]:
def exercise_3_fib_matrix(n: int) -> int:
    # TODO: trả về F(n) mod 10^9+7 bằng lũy thừa ma trận [[1,1],[1,0]]
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")


# assert exercise_3_fib_matrix(10) == 55

## Bài 4 — Climbing Stairs với `n` khổng lồ

Mỗi bước leo 1 hoặc 2 bậc. Số cách leo `n` bậc (mod `10^9 + 7`) với `n <= 10^18`.

```python
climb(2) -> 2, climb(3) -> 3, climb(10) -> 89
```

In [20]:
def exercise_4_climb(n: int) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_4_climb(10) == 89

## Bài 5 — Mọi run có độ dài **không chia hết cho 3**? Đếm bằng DP `O(n)`

Đếm số chuỗi `a/b` độ dài `n` mà mọi run có độ dài không chia hết cho 3 (`n <= 1000`). Kiểm tra với brute force.

In [21]:
def exercise_5_runs_not_mult3(n: int) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_5_runs_not_mult3(3) == 6

## Bài 6 — Mini challenge: bảng chữ cái `k` ký tự

Đếm số chuỗi độ dài `n` trên bảng chữ cái `k` ký tự mà mọi run có độ dài lẻ (mod `10^9+7`, `n <= 10^15`). Gợi ý: gọi `G(n)` = số cách với ký tự đầu cố định; tìm truy hồi bậc 2 rồi dùng lũy thừa ma trận. Kiểm tra với brute force cho `k = 3`.

In [22]:
def exercise_6_k_letters(n: int, k: int) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_6_k_letters(4, 2) == 6

## Gợi ý (mở từng mức)

**Bài 3**

- Mức 1: viết `mat_mult` cho ma trận `2×2` có lấy dư.
- Mức 2: lũy thừa bằng bình phương liên tiếp, bắt đầu từ ma trận đơn vị.
- Mức 3: `F(n)` là phần tử `[0][1]` của `T^n`.

**Bài 4**

- Mức 1: số cách leo `n` bậc thỏa `a(n) = a(n-1) + a(n-2)`, `a(1) = 1`, `a(2) = 2`.
- Mức 2: `a(n) = F(n + 1)`.
- Mức 3: `fib_pair(n + 1)[0]`.

**Bài 5**

- Mức 1: `D(m)` = số cách viết `m` thành tổng các số không chia hết cho 3.
- Mức 2: `D(m) = Σ_{r không chia hết 3} D(m - r)`; giữ thêm tổng tiền tố để `O(n)`.
- Mức 3: đáp án `2 · D(n)`.

**Bài 6**

- Mức 1: run cuối dài 1 → phần trước có `n - 1` ký tự, run cuối có `k - 1` lựa chọn ký tự.
- Mức 2: run cuối dài `>= 3` → giảm 2, giữ nguyên ký tự → `G(n - 2)`. Vậy `G(n) = (k-1)·G(n-1) + G(n-2)`.
- Mức 3: `G(1) = 1`, `G(2) = k - 1`; đáp án `k · G(n)`; ma trận `[[k-1, 1], [1, 0]]`.

# 17. Đáp án tham khảo

## Đáp án Bài 1

8 cách: `1+1+1+1+1+1`, `1+1+1+3`, `1+1+3+1`, `1+3+1+1`, `3+1+1+1`, `3+3`, `1+5`, `5+1`. Đúng bằng `F(6) = 8`; đáp án `2 · 8 = 16`.

## Đáp án Bài 2

```text
[1, 2, 1] False
[[8, 5], [5, 3]]
(55, 89)
```

In [23]:
# Đáp án Bài 3
def answer_3_fib_matrix(n: int) -> int:
    return mat_pow([[1, 1], [1, 0]], n)[0][1]


assert answer_3_fib_matrix(10) == 55
assert answer_3_fib_matrix(10**15) == fib_pair(10**15)[0]
print("Bài 3: PASS")

Bài 3: PASS


In [24]:
# Đáp án Bài 4
def answer_4_climb(n: int) -> int:
    return fib_pair(n + 1)[0]


assert [answer_4_climb(n) for n in (2, 3, 10)] == [2, 3, 89]
print("Bài 4: PASS | climb(10^18) =", answer_4_climb(10**18))

Bài 4: PASS | climb(10^18) = 680057396


In [25]:
# Đáp án Bài 5
def answer_5_runs_not_mult3(n: int) -> int:
    D = [0] * (n + 1)
    D[0] = 1
    for m in range(1, n + 1):
        D[m] = sum(D[m - r] for r in range(1, m + 1) if r % 3) % MOD
    return 2 * D[n] % MOD


def brute_not_mult3(n):
    return sum(all(r % 3 for r in run_lengths("".join(p))) for p in product("ab", repeat=n))


for n in range(1, 11):
    assert answer_5_runs_not_mult3(n) == brute_not_mult3(n)
assert answer_5_runs_not_mult3(3) == 6
print("Bài 5: PASS")

Bài 5: PASS


In [26]:
# Đáp án Bài 6
def answer_6_k_letters(n: int, k: int) -> int:
    if n == 1:
        return k % MOD
    T = mat_pow([[k - 1, 1], [1, 0]], n - 2)
    G2, G1 = (k - 1) % MOD, 1
    Gn = (T[0][0] * G2 + T[0][1] * G1) % MOD
    return k * Gn % MOD


def brute_k(n, k):
    letters = "abcdefg"[:k]
    return sum(all(r % 2 for r in run_lengths("".join(p))) for p in product(letters, repeat=n))


for k in (2, 3):
    for n in range(1, 8):
        assert answer_6_k_letters(n, k) == brute_k(n, k), (n, k)
assert answer_6_k_letters(4, 2) == 6
print("Bài 6: PASS (khớp brute force với k = 2, 3)")

Bài 6: PASS (khớp brute force với k = 2, 3)


# 18. Tổng kết

**Count Good Strings**:

1. Liệt kê nhỏ → đặc trưng: **chuỗi tốt ⇔ mọi run có độ dài lẻ**.
2. Đếm = `2 ×` (số cách viết `n` thành tổng có thứ tự các số lẻ) = `2 · F(n)`.
3. `n <= 10^15` → lũy thừa ma trận / fast doubling `O(log n)`.

## Độ phức tạp

```text
Time:  O(log n)
Space: O(log n)
```

# 19. Checklist trước khi submit

- [ ] Đáp án là `2 · F(n)` với `F(1) = F(2) = 1` (kiểm tra `n = 2, 3, 4`)?
- [ ] Lấy dư `10^9 + 7` ở mọi phép nhân/cộng?
- [ ] Dùng thuật toán `O(log n)` (không vòng lặp `n` lần)?
- [ ] Xử lý `n = 1` (đáp án 2)?